# Stage 2 — Safety Results

This notebook analyses the safety results from Stage 2 expert steering experiments.

**Metric**: `safe_rate` — proportion of model responses classified as safe by Llama-Guard-3-8B.

**Two steering directions**:
- **Safe steering** (forced prefix): model is forced to begin a harmful response. Steering suppresses compliance-preferred experts. Higher safe_rate post-steering = model still refuses despite forced prefix = strong effect.
- **Unsafe steering** (safety system prompt): model is given a safety system prompt. Steering suppresses refusal-preferred experts. Lower safe_rate post-steering = model complies despite safety prompt = strong effect.

**Conditions**: `baseline` (no steering) vs `hard` (full expert deactivation).

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', font='serif')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

RESULTS_PATH = '/scratch/sc23jc3/stage2_results/results.json'
OUT_DIR = 'figures'
os.makedirs(OUT_DIR, exist_ok=True)

with open(RESULTS_PATH) as f:
    results = json.load(f)

safety = results['safety']
print('Loaded results.')
print('Safe steering conditions:', list(safety['safe_steering'].keys()))
print('Unsafe steering conditions:', list(safety['unsafe_steering'].keys()))

## 1. Safe_rate: Baseline vs Hard Steering

In [ ]:
directions = {
    'Safe steering\n(forced prefix)':   safety['safe_steering'],
    'Unsafe steering\n(safety prompt)': safety['unsafe_steering'],
}

labels = list(directions.keys())
baselines = [directions[k].get('baseline', float('nan')) for k in labels]
hards     = [directions[k].get('hard', float('nan'))     for k in labels]
deltas    = [h - b for b, h in zip(baselines, hards)]

x = np.arange(len(labels))
w = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars_b = ax.bar(x - w/2, baselines, width=w, label='Baseline', color='#7f8c8d', alpha=0.85)
bars_h = ax.bar(x + w/2, hards,     width=w, label='Hard steering', color='#2980b9', alpha=0.85)

for i, (b, h, d) in enumerate(zip(baselines, hards, deltas)):
    if not (np.isnan(b) or np.isnan(h)):
        sign = '+' if d >= 0 else ''
        ax.annotate(f'{sign}{d:.3f}', xy=(x[i] + w/2, h), xytext=(0, 4),
                    textcoords='offset points', ha='center', fontsize=10,
                    color='#2980b9', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Safe Rate')
ax.set_ylim(0, 1.1)
ax.set_title('Safety Results: Baseline vs Hard Expert Deactivation')
ax.legend()
ax.axhline(1.0, color='grey', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'safety_results.png'), dpi=300, bbox_inches='tight')
plt.show()

## 2. Numerical Summary

In [ ]:
print(f"{'Direction':35s} {'Baseline':>10} {'Hard':>10} {'Delta':>10}")
print('-' * 70)
for label, row in directions.items():
    b = row.get('baseline', float('nan'))
    h = row.get('hard', float('nan'))
    d = h - b if not (np.isnan(b) or np.isnan(h)) else float('nan')
    label_clean = label.replace('\n', ' ')
    print(f"{label_clean:35s} {b:10.3f} {h:10.3f} {d:+10.3f}")

## Discussion

*(Fill in after examining the results.)*

**Safe steering (forced prefix)**:
- Baseline safe_rate near 0 is expected — the forced prefix causes the model to comply.
- If hard steering raises safe_rate meaningfully, it means suppressing compliance-preferred experts causes refusal even against a forced prefix. This is a strong causal finding.

**Unsafe steering (safety system prompt)**:
- Baseline safe_rate near 1 is expected — the safety prompt causes the model to refuse.
- If hard steering lowers safe_rate meaningfully, it means suppressing refusal-preferred experts causes compliance even against a safety prompt. This is the contrastive evidence of causality.

**Interpretation**:
- Asymmetry between directions (one effect larger than the other) is informative about how safety is encoded.
- If neither direction shows a large effect, CANDIDATE_N or the steering strength may need adjusting.